In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


### **1. Basic Data Checking**

In [ ]:
# Download data
df = pd.read_csv("../data/raw/train.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
# Change Dtype

df['CustomerID'] = df['CustomerID'].astype(int).astype(str)

# transform Churn into int 
df['Churn'] = df['Churn'].astype(int)

# transform Age into int 
df['Age'] = df['Age'].astype(int)

# Category
category_cols = ['Gender', 'Subscription Type', 'Contract Length']

for col in category_cols:
    df[col] = df[col].astype('category')

In [ ]:
# numeric columns descriptive statistic
## values are valid

df.describe().round(2)

In [ ]:
# category columns

gender = df['Gender'].value_counts()
sub_type = df['Subscription Type'].value_counts()
contract = df['Contract Length'].value_counts()

print(gender,'\n\n')
print(sub_type, '\n\n')
print(contract)

In [ ]:
# missing value 

print(f"Total null value: {df.isnull().sum().sum()}\n")
print("Total missing value for each columns:")
df.isnull().sum()

In [ ]:
# Drop CustomerID
df = df.drop(columns='CustomerID')
df.head()

### **2. Variable Description**

- Source: https://www.kaggle.com/datasets/muhammadshahidazeem/customer-churn-dataset/data
- Total number of observations (rows): 440,831
- Total number of attributes (columns): 11
- Target Variables: `Churn` (Binary class)
- Numerical Features: `Age`, `Tenure`, `Usage Frequency`, `Support Calls`, `Payment Delay`, `Total Spend`, `Last Interaction`
- Category Features: `Gender`, `Subscription Type`, `Contract Length`


| Variable Name | Description | Example (Value & Unit) |
| :--- | :--- | :--- |
| **Age** | The age of the customer. | `28` (Years) |
| **Gender** | The biological sex of the customer. | `Male` / `Female` |
| **Tenure** | Duration of the customer's relationship with the company. | `12` (Months) |
| **Usage Frequency** | How often the customer uses the service per month. | `15` (Times/Month) |
| **Support Calls** | The number of times the customer contacted customer support center. | `4` (Calls) |
| **Payment Delay** | Number of days the customer is late on their billing. | `7` (Days) |
| **Subscription Type** | The specific service tier chosen by the customer. | `Basic`, `Standard`, `Premium` |
| **Contract Length** | The commitment period of the service agreement. | `Monthly`, `Quarterly`, `Annual` |
| **Total Spend** | Total revenue generated from the customer to date. | `1250.50` (Currency Units) |
| **Last Interaction** | Days elapsed since the last customer engagement. | `3` (Days ago) |
| **Churn** | Target variable indicating if the customer has left. | `1` (Yes) or `0` (No) |

### **3. Target Variable and Metrics**

The class distribution (43.3% No / 56.7% Yes) is reasonably balanced, eliminating concerns about severe bias and allowing standard evaluation metrics (Accuracy, F1, ...) to reliably reflect model performance.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

sns.countplot(data=df, x="Churn", ax=ax, palette="pastel") 

plt.title("The distribution for Churn")

counts = df['Churn'].value_counts().sort_index()
pct = (counts / counts.sum() * 100).round(1)

colors = [p.get_facecolor() for p in ax.patches]

handles = [
    Patch(facecolor=colors[0], label=f'No (0): {pct.get(0, 0):.1f}%'),
    Patch(facecolor=colors[1], label=f'Yes (1): {pct.get(1, 0):.1f}%')
]

ax.legend(handles=handles, loc='upper right', bbox_to_anchor=(1.5, 1))
fig.subplots_adjust(right=0.75) 

plt.show()

##### **Metrics Suggestion**
#### **3.1. Accuracy**

- Definition: The proportion of total correct predictions (both classes) among the total number of cases examined.

- Usage: Used as a baseline, but can be misleading in imbalanced datasets like ours.

- Formula:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

#### **3.2. Precision (Positive Predictive Value)**
- Meaning: Among the customers that the model predicts as "Will Churn", what percentage actually churned?  

- Why use it: Helps businesses optimize marketing costs. By maximizing Precision, you avoid wasting resources on retention campaigns (e.g., gifts, discounts, special offers) for customers who were actually loyal and had no intention of leaving.

- Formular:

$$\text{Precision} = \frac{TP}{TP + FP}$$


#### **3.3. Recall**

- Meaning: Among all customers who actually churned, what percentage did the model correctly identify?  

- Why use it: This is the most critical metric for Churn Prediction problems. Businesses often adopt a "better safe than sorry" approach – it is preferable to incorrectly offer a retention voucher to a loyal customer than to miss a high-risk customer and lose them to a competitor.

- Formular:

$$\text{Recall} = \frac{TP}{TP + FN}$$


#### **3.4. F1-Score**

- Meaning: The harmonic mean of Precision and Recall. It provides a single metric that balances both concerns.

- Why use it: Especially useful when dealing with imbalanced datasets. A high F1-Score indicates the model effectively identifies positive cases (high Recall) while minimizing false alarms (high Precision).

- Formula:
$$
\text{F1-score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}
$$



#### **3.5. Aggregating F1-Score for Multi-Class Problems**

When evaluating models with multiple classes (or binary classification with detailed reporting), we need to aggregate the F1-Scores from each class. Two common methods are Macro-Average and Weighted-Average.

##### **F1-Macro (Unweighted Mean)**

- Meaning: Calculates the F1-Score for each class independently, then takes their simple arithmetic mean. It treats all classes as equally important, regardless of their sample size.

- Formula:
$$
\text{F1-Macro} = \frac{1}{N} \sum_{i=1}^{N} \text{F1}_i
$$
*(Where $N$ is the number of classes)*

- When to use: 
  - When you care about performance on **minority classes** as much as majority ones.
  - When you want to detect if the model is biased towards the majority class.

##### **F1-Weighted (Support-Weighted Mean)**

- Meaning: Calculates the F1-Score for each class, then computes the average weighted by the number of true instances (support) for each class.

- Formula:
$$
\text{F1-Weighted} = \sum_{i=1}^{N} \left( \text{F1}_i \cdot \frac{\text{Support}_i}{\text{Total Samples}} \right)
$$

- When to use: 
  - When dataset is imbalanced and you want the metric to reflect overall performance across the entire dataset.
  - When the cost of errors is proportional to class frequency.



### **4. Feature Distribution**

#### **4.1. Numerical Features**

##### **General Observations**
- **Uniformity:** Features like `Tenure`, `Usage Frequency`, and `Payment Delay` exhibit a near-uniform distribution across their ranges. This could be due to synthetic dataset.
- **No Outliers:** Based on the boxplots and the Interquartile Range (IQR), there are no significant statistical outliers in any numerical columns. All values fall within logical business bounds.
- **Symmetry:** Most features show a high degree of symmetry, with mean and median (50%) values being very close, indicating minimal skewness.

#### **Specific Feature Insights**
- **Age:** There is a noticeable drop in the customer count for the 50-65 age group compared to younger segments.
- **Support Calls:** Follows a right-skewed (decreasing) discrete distribution. The majority of customers make 0-3 calls, while very few reach the maximum of 10 calls. This is likely a key driver for Churn.
- **Payment Delay:** Distribution is uniform up to 20 days, followed by a sharp structural decline. Customers with >20 days delay are significantly fewer in number.
- **Total Spend:** There is a significantly higher concentration of customers spending between $500 - $1000 compared to the lower spending bracket ($100 - $500).
- **Last Interaction:** Shows a uniform engagement pattern for the first 15 days, which then drops to a lower, steady plateau for the remaining 16-30 days.

#### **Suggestions**
- **Feature Scaling:** Since features have vastly different ranges (e.g., `Support Calls` [0-10] vs. `Total Spend` [100-1000]), Scaling (Standardization/Normalization) is mandatory for distance-based models.
- **Discretization Potential:** Features with sharp drops (like `Payment Delay` at 20 days) could be candidates for binning to capture non-linear effects during feature engineering.

In [ ]:
numerical_features = [
    col for col in df.columns if df[col].dtype != 'category' and col != "Churn"
]

df[numerical_features].describe().round(2)

In [ ]:
sns.set_style("white") 

fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes_list = axes.flatten()

for feature, ax in zip(numerical_features, axes_list):
    sns.histplot(data=df, x=feature,color = '#4C72B0' ,kde=True, ax=ax)
    ax.set_title(f"Distribution of {feature}")

axes_list[7].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(20, 10))
axes_list = axes.flatten()

for feature, ax in zip(numerical_features, axes_list):
    sns.boxplot(data=df, x=feature, ax=ax)
    ax.set_title(f"Distribution of {feature}")

axes_list[7].axis("off")
plt.tight_layout()
plt.show()

#### **4.2. Category Features**

- **Gender**: The dataset has a slight imbalance between genders. Male customers account for 57%, while Female customers make up 43%.

- **Subscription Type**: Distribution is also balanced across all three tiers: Basic (32%), Premium (34%), and Standard (34%). 

- **Contract Length**: There is a notable gap in contract preferences. Both Annual and Quarterly contracts are dominant, each representing 40% of the user base. Interestingly, Monthly contracts are the least popular, accounting for only 20%.



In [ ]:
categorical_features = [col for col in df.columns if df[col].dtype == 'category']

sns.set_style("white")
sns.set_palette("pastel")

fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6))
axes_list = axes.flatten()

for feature, ax in zip(categorical_features, axes_list):
    plot = sns.countplot(data=df, x=feature, ax=ax, color= "#9BC4DA")
    ax.set_title(f"Distribution of {feature}", fontsize=14, fontweight='bold')
    
    total = len(df[feature].dropna()) 
    
    for p in plot.patches:
        if p.get_height() > 0:  
            height = p.get_height()
            pct = (height / total) * 100
            label = f"{pct:.0f}%" if pct >= 10 else f"{pct:.1f}%"
            ax.annotate(
                label,
                xy=(p.get_x() + p.get_width() / 2, height), 
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom',
                fontsize=10, fontweight='bold'
            )
    
    ax.grid(axis='y', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

### **5. Feature Relationship**

#### **5.1. Numerical and Target Variables**

#### Important 
- **Support Calls:** This is the most distinctive feature. The median for Churners is significantly higher (~5 calls) compared to Non-churners (~1 call). High interaction with support is a clear red flag.

- **Total Spend:** There is a stark contrast here. Customers who stay have a much higher median spend (~$750$) than those who churn (~$500$). Lower spending behavior is strongly correlated with attrition.

- **Payment Delay:** Churners show a higher median delay and a much wider interquartile range (IQR). As seen in previous binning analysis, delays exceeding 20 days lead to 100% churn.

- **Last Interaction:** Customers who churn tend to have their last interaction longer ago (higher median) compared to loyal customers.

#### Moderate Features
- **Age:** Churners are generally older (median ~42) than those who stay (median ~37). While there is overlap, the shift in the distribution is statistically visible.

#### Low Importance Features
- **Tenure:** The boxplots for both groups are almost identical in terms of median (~30 months) and range. This suggests that how long a customer has been with the company does not directly influence their decision to leave in this specific dataset.
- **Usage Frequency:** Similar to Tenure, the distributions heavily overlap. The median usage is roughly the same for both groups (~15-17 times), making it a weak predictor for the model.


In [ ]:
plt.figure(figsize=(16, 12))
plt.suptitle("Relationship Between Numeric Features and Churn", fontsize=18, y=1.02)

for i, col in enumerate(numerical_features):
    plt.subplot(2, 4, i + 1)
    sns.boxplot(data=df, x="Churn", y=col, palette="pastel")

    plt.title(f"{col} vs. Churn")
    plt.xlabel("Churn (0 = No, 1 = Yes)")
    plt.ylabel(col)

plt.tight_layout(rect=(0, 0.03, 1, 0.98))
plt.show()

#### **Binning Age group and Churn**
- Group Aldult (25 - 39 years old) has the lowest churn rate, while Senior (> 60 years old) has 100% churn rate
- The two remaining group Young Aldult (18-24) and Mid-Career (40-59) has similar churn rate (~ 55%)


In [ ]:
# 1. Create bins
bins = [18, 24, 39, 59, df['Age'].max()]
labels = ['18-24', '25-39', '40-59', '> 60']

# 2. Create Age_Group
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, include_lowest=True)

# 3. Calculate Churn rate
age_churn_rate = df.groupby('Age_Group', observed=True)['Churn'].mean().reset_index()
age_churn_rate['Churn_Percentage'] = age_churn_rate['Churn'] * 100

# 4. Draw figure
plt.figure(figsize=(7, 4))
sns.set_style("white")

# Color setup
custom_palette = []
for label in labels:
    if label == "Senior":
        custom_palette.append("#4d99d0") 
    else:
        custom_palette.append("#a1c9f4") 

ax = sns.barplot(data=age_churn_rate, x='Age_Group', y='Churn_Percentage', 
                 palette=custom_palette)

# Thêm con số phần trăm cụ thể trên đầu cột
for p in ax.patches:
    height = p.get_height()
    if height > 0: 
        ax.annotate(f'{height:.1f}%', 
                    (p.get_x() + p.get_width() / 2., height), 
                    ha='center', va='bottom', 
                    fontsize=12, fontweight='bold', xytext=(0, 5),
                    textcoords='offset points')

plt.title('Churn Rate by Age Group (Highlighting Senior)', fontsize=15, fontweight='bold')
plt.xlabel('Age Group', fontsize=12)
plt.ylabel('Churn Rate (%)', fontsize=12)
plt.ylim(0, age_churn_rate['Churn_Percentage'].max() + 10) 

plt.show()

#### **Payment Delay and Churn**

- Critical Threshold at 20 Days: There is a dramatic structural shift once payment delay exceeds **20 days**. The count of active records drops significantly (left plot), while the churn rate spikes to an absolute **100.0%** (right plot).
- For customers with a delay between **0 and 20 days**, the churn rate remains perfectly stable at **46.5%**.


In [ ]:
sns.set_palette("pastel")
sns.set_style("white")

# Create figure with 2 columns
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(17, 8))

# ============================================================
# 📊 PLOT 1: Distribution of Payment Delay (Histogram + KDE)
# ============================================================
sns.histplot(data=df, x='Payment Delay', kde=True, ax=axes[0], 
             color='skyblue', edgecolor='black', linewidth=0.8)
axes[0].set_title("Distribution of Payment Delay", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Payment Delay (days)")
axes[0].set_ylabel("Count")
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# ============================================================
# PLOT 2: Churn Rate by Payment Delay Bins
# ============================================================
# Binning variable Payment Delay
df['Delay_Bin'] = pd.cut(df['Payment Delay'], 
                         bins=[0, 10, 20, 30], 
                         labels=['0-10 days', '11-20 days', '21-30 days'],
                         include_lowest=True)

# Caluculate churn rate by each bin
churn_by_bin = df.groupby('Delay_Bin', observed=False)['Churn'].mean().reset_index()

# Draw barplot with axes[1]
sns.barplot(data=churn_by_bin, x='Delay_Bin', y='Churn', ax=axes[1], 
            palette='magma', edgecolor='black')
axes[1].set_title("Churn Rate by Payment Delay Bins", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Payment Delay Range")
axes[1].set_ylabel("Churn Rate")
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

# Add % label 
for p in axes[1].patches:
    if p.get_height() > 0:
        height = p.get_height()
        axes[1].annotate(f'{height*100:.1f}%', 
                        xy=(p.get_x() + p.get_width()/2, height),
                        xytext=(0, 3), textcoords="offset points",
                        ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

#### **5.2. Categorical and Target Variables**

- **Contract Length**: Customers with Contract Type Monthly have a 100% Churn Rate. Meanwhile, types Annual and 2.0 (Quarterly) have much lower and identical churn rates (~45%). Thus, Monthly contracts could be a significant predictor of churn. 

- **Gender**: There is a visible difference between genders. Female has a higher churn rate (~65%) compared to Male (~50%). Gender carries a strong signal in this dataset and should be prioritized during feature selection.

- **Subscription Type**: The churn rate across all three subscription tiers is virtually identical (approx. 55-58% churn). Because the proportion of churn does not change regardless of whether a customer is on a Basic, Standard, or Premium plan, this feature provides zero discriminative power for the model.


In [ ]:
plt.figure(figsize=(18, 6))
plt.suptitle("Relationship Between Categorical Features and Churn", fontsize=18, y=1.03)

df_plot = df.copy()
df_plot["Churn"] = df_plot["Churn"].map({0.0: "No Churn", 1.0: "Churn"})

for i, col in enumerate(categorical_features):
    plt.subplot(1, 3, i + 1)

    sns.histplot(
        data=df_plot,
        x=col,
        hue="Churn",  # Color-code the bars by 'Churn' status
        multiple="fill",  # Stack bars and normalize to 100% (fill)
        stat="proportion",  # Show proportions (not raw counts)
        discrete=True,  # Treat the x-axis as discrete categories
        shrink=0.8,  # Add a small gap between bars
    )

    plt.title(f"Churn Rate by {col}")
    plt.xlabel(col)
    plt.ylabel("Proportion of Customers")

    if i == 0:
        sns.move_legend(plt.gca(), "upper left", bbox_to_anchor=(-0.25, 1))
    else:
        plt.gca().get_legend().remove()

plt.tight_layout(rect=(0, 0.03, 1, 0.98))
plt.show()

#### **5.3. Correlation Heatmap**

In [ ]:
ordinal_encoder = OrdinalEncoder()
label_encoder = LabelEncoder()

In [ ]:
df["Subscription Type"] = ordinal_encoder.fit_transform(
    df["Subscription Type"].values.reshape(-1, 1)
)

# Although 'Monthly' shows a churn rate of 1, its distribution accounts for only ~20% of the dataset.
# The remaining two contract lengths (80%) provide little class distinction, resulting in a near-zero correlation (~0) with Churn.
df["Contract Length"] = ordinal_encoder.fit_transform(
    df["Contract Length"].values.reshape(-1, 1)
)

# df['is_monthly_contract'] = (df['Contract Length'] == 'Monthly').astype(int)

df["Gender"] = label_encoder.fit_transform(df["Gender"].values.reshape(-1, 1))
df.head()

In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64",'int32'])
numeric_df.columns


In [ ]:
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,  # Display the correlation values on the heatmap
    fmt=".2f",  # Format the values to two decimal places
    cmap="coolwarm",  # Use a diverging colormap (blue=negative, red=positive)
    linewidths=0.5,  # Add fine lines between cells
    vmin=-1,  # Set the min of the colormap to -1
    vmax=1,  # Set the max of the colormap to +1
)

plt.title("Correlation Heatmap of Numeric Features", fontsize=16)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

- `Support Calls` and `Total Spend` have the highest correlation with Churn, while features like `Tenure`, `Usage Frequency`, `Subscription Type`, `Contract Length` show nearly no linear relationship. 

- For the case of `Contract Length`, enventhough the Monthly Contract (only account for 20% dataset) show the 100% churn rate, the other two types (Annual and Quarterly) have nearly identical churn rate. Additionally, because Monthly contracts result in a 100% churn rate, the target variable becomes a constant within that specific group (zero variance). Binary feature like `is_monthly_contract` will also lead to near-zero results in Pearson correlation matrices. 

- As for final selected features, we will only choose `Age`, `Gender`, `Support Calls`, `Payment Delay`, `Total Spend` and `Last Interaction`. 